In [1]:
! python3 -m pip install pandas seaborn numpy matplotlib scikit-learn xgboost scipy "spacy==3.7.5" imbalanced-learn tensorflow

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import seaborn as sns 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix, roc_auc_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
from tsai.all import *
from imblearn.over_sampling import SMOTE

/Users/tommy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
!python3 -m pip install tsai

Defaulting to user installation because normal site-packages is not writeable


In [4]:
dataset = pd.read_csv("./Dataset.csv")

In [5]:
dataset.head(5)

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [6]:
dataset.columns

Index(['Unnamed: 0', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP',
       'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
       'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
       'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
       'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC',
       'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2',
       'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Patient_ID'],
      dtype='object')

## Preprocessing the data : add padding, drop, split, input and scale

In [7]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [8]:
numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value= 0)), ("scaler", StandardScaler())]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value = "UNKNOWN")), # no data leak hehe
        ("encoder", LabelEncoder()),
    ]
)

cat_cols = []
num_cols = []
for col_name, dtype in dataset.dtypes.items():

    if col_name == "SepsisLabel" or col_name == "Patient_ID":
        continue  

    if np.issubdtype(dtype, np.number):
        num_cols.append(col_name)
    else:
        cat_cols.append(col_name)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols,
        ),
        
    ],
    remainder="passthrough"
)

columns = dataset.columns

data_processed = preprocessor.fit_transform(dataset)
dataset_reconstruct = pd.DataFrame(data_processed, columns= columns)

In [9]:
dataset_reconstruct.shape

(1552210, 44)

In [10]:
groups = dataset_reconstruct.groupby("Patient_ID")

hours =[]

transformed_sequence = []

# Analysis of the patient time lentgh
for patient_id, patient_df in groups:
    hours.append(patient_df.shape[0])

window = np.percentile(hours,95) # 95% percentile
window


58.0

In [11]:
# preprocessing : drop all > 95% percentile hours and add padding
columns_transformed = None

target_var = []

for patient_id, patient_df in groups:

    #extract the target variable
    # if the patient have sepsis, they will have at least 1 row to be 1
    label = 0
    for i in patient_df["SepsisLabel"]:
        if i == 1:
            label = 1

    patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])

    if patient_df.shape[0] > window:
        #im lazy sorry:(
        columns_transformed = patient_df.columns
        continue

    target_var.append(label)
    #add padding
    patient_df = patient_df.sort_values("Hour")
    
    num_feature = patient_df.shape[1] # should be 43 anyway


    #change to np
    patient_np = patient_df.to_numpy()
    num_padding_rows = int(window) - patient_df.shape[0]
    all_zeros = np.zeros(num_feature)
    new_patient = []
    for i in range (num_padding_rows):
        new_patient.append(all_zeros)
    for hour in patient_np:
        new_patient.append(hour)

    # new_patient_np = pd.DataFrame(
    #     np.asarray(new_patient),
    #     columns=patient_df.columns
    # )
    new_patient_np = np.asarray(new_patient)

    transformed_sequence.append(new_patient)

transformed_sequence = np.asarray(transformed_sequence)
print(transformed_sequence.shape)

(38617, 58, 42)


In [12]:
transformed_sequence[0:5]

array([[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 8.83137308e-01,  8.83137308e-01,  2.91347620e-01, ...,
         -6.62381181e-01,  3.45717042e-01,  8.62080829e-01],
        [ 9.17760293e-01,  9.17760293e-01,  3.24540480e-01, ...,
         -6.62381181e-01,  3.45717042e-01,  8.96557156e-01],
        [ 9.52383277e-01,  9.52383277e-01,  2.58154760e-01, ...,
         -6.62381181e-01,  3.45717042e-01,  9.31033483e-01]],

       [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e

In [13]:
target_var

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,


In [14]:
target_var_np = np.asarray(target_var)

In [15]:
target_var_np.shape

(38617,)

In [16]:
# Split train test
X = transformed_sequence
y = target_var
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size= 0.3, random_state= 67, stratify=y)

print(X_train.shape)
print(X_test.shape)
print(len(y_train))
print(len(y_test))

(27031, 58, 42)
(11586, 58, 42)
27031
11586


In [17]:
print(len([one for one in y_train if one == 0]))

25703


In [18]:
#SMOTE sampling X_train

#Flatten to 2D

num_sample, num_time, num_feature = X_train.shape

X_train_flatten = X_train.reshape(num_sample, num_time*num_feature) #Flatten to do SMOTE
#SMOTE time
s = SMOTE(k_neighbors=10, random_state= 67)
X_train_sampled_flatten , y_train_sampled = s.fit_resample(X_train_flatten,y_train)
X_train_sampled = X_train_sampled_flatten.reshape(-1, num_time, num_feature) # reshape back to 3D

X_train_sampled.shape

(51406, 58, 42)

In [19]:
print(len(y_train_sampled))

print(len([one for one in y_train_sampled if one == 1]))
print(len([one for one in y_train_sampled if one == 0]))

51406
25703
25703


In [20]:
X_train_sampled = np.asarray(X_train_sampled, dtype=np.float32)
y_train_sampled = np.asarray(y_train_sampled, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

### Baseline: RNN (simple small) - Note: No pass this baseline on Recall Sepsis -> Try again

In [21]:
# Baseline SimpleRNN for the patient sequences
import random
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

SEED = 67
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

num_time_steps, num_features = X_train_sampled.shape[1], X_train_sampled.shape[2]

model = Sequential([
    SimpleRNN(32, input_shape=(num_time_steps, num_features)),
    Dense(1, activation="sigmoid"),
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    X_train_sampled,
    y_train_sampled,
    epochs=10,
    batch_size=100,
    verbose=2,
)

test_probability = model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability >= 0.5).astype(np.int32)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0,
))
print(f"ROC-AUC: {roc_auc_score(y_test, test_probability):.4f}")

Epoch 1/10


2026-10-05 16:30:33.466751: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


515/515 - 2s - 4ms/step - accuracy: 0.8585 - loss: 0.3274
Epoch 2/10
515/515 - 1s - 2ms/step - accuracy: 0.9197 - loss: 0.2116
Epoch 3/10
515/515 - 1s - 2ms/step - accuracy: 0.9352 - loss: 0.1736
Epoch 4/10
515/515 - 1s - 2ms/step - accuracy: 0.9444 - loss: 0.1508
Epoch 5/10
515/515 - 1s - 2ms/step - accuracy: 0.9486 - loss: 0.1392
Epoch 6/10
515/515 - 1s - 2ms/step - accuracy: 0.9530 - loss: 0.1288
Epoch 7/10
515/515 - 1s - 2ms/step - accuracy: 0.9563 - loss: 0.1196
Epoch 8/10
515/515 - 1s - 2ms/step - accuracy: 0.9550 - loss: 0.1245
Epoch 9/10
515/515 - 1s - 2ms/step - accuracy: 0.9593 - loss: 0.1121
Epoch 10/10
515/515 - 1s - 2ms/step - accuracy: 0.9623 - loss: 0.1049


2026-10-05 16:30:46.448266: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


              precision    recall  f1-score   support

   No sepsis     0.9787    0.9473    0.9627     11017
      Sepsis     0.3705    0.6011    0.4584       569

    accuracy                         0.9303     11586
   macro avg     0.6746    0.7742    0.7106     11586
weighted avg     0.9488    0.9303    0.9380     11586

ROC-AUC: 0.8853
